# AdaptSynth pipeline: any dataset × any generator

One notebook for the whole experiment. Change the two lines in **Settings** to run a different dataset or generator.

```
real data ──split──► TRAIN (70%)                              TEST (30%, only used for scoring)
                       │
             data preparation  (drop correlated/constant columns, impute, optional outlier filter)
                       │
        ┌──────────────┴───────────────┐
        │ 5 feature-ranking techniques │  filter: ANOVA, chi-square, mutual info | wrapper: RFE | embedded: RF importance
        └──────────────┬───────────────┘
                       ▼
          voting → final top-N features
                       │
       ┌───────────────┴────────────────┐
       ▼                                ▼
 each generator on ALL features     each generator on TOP-N features
       │                                │
       └───────► synthetic tables ◄─────┘
                       │
   score against the REAL TEST table:
   (A) column distributions  (B) column relationships  (C) TSTR / TRTR utility (XGBoost: macro-F1 + ROC-AUC)
```

| Datasets (`DATASET=`) | Generators (`GENERATORS=[...]`) |
|---|---|
| `iot23` – IoT-23 Scenario 1 (Mirai) | `ctabgan_plus` – CTAB-GAN+ (official code) |
| `sdn` – SDN DDoS (chronological split) | `ctgan` – CTGAN |
| `guide` – Microsoft GUIDE (all-categorical) | `tvae` – TVAE |
| `cic_ids2017` – CIC-IDS2017 (sampled) | `smote` – SMOTE / SMOTENC |

**Rule that keeps the experiment honest:** feature selection, imputation and every other statistic only ever see the TRAIN split.

## 0. Settings

In [ ]:
import sys, json, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

PROJECT = Path.cwd()
if not (PROJECT / "src" / "adaptsynth").exists():
    raise FileNotFoundError("Start this notebook from the AdaptSynth project directory.")
sys.path.insert(0, str(PROJECT / "src"))
import adaptsynth as A

# ------------------------------------------------------------------ what to run
DATASET    = "iot23"            # "iot23" | "sdn" | "guide" | "cic_ids2017"
GENERATORS = ["ctabgan_plus"]   # add any of: "ctgan", "tvae", "smote"   (adding your own model: see the last section)
EPOCHS     = 200                # training epochs for the neural generators (SMOTE ignores it)
SEEDS      = [42]               # use [42, 43, 44] to report mean ± std
N_SELECT   = None               # features kept after voting; None = dataset default (iot23 6, sdn 6, guide 6, cic 10)

# ------------------------------------------------------------------ data preparation (paper Section IV-B)
CORR_THRESHOLD  = 0.95          # drop one of each pair of numeric features correlated above this
REMOVE_OUTLIERS = False         # IQR outlier filter on TRAIN rows. OFF by default, see section 2
ZSCORE          = False         # z-score the numeric features

LOG_TO_PAPER_RESULTS = True     # append to paper_results/experiment_log.csv and rebuild the Excel workbook

## 1. Load the dataset

Each loader does only the dataset-specific cleaning (drop identifiers, choose columns, sample rows). The target column is always called `label`.

In [ ]:
ds = A.datasets.load(DATASET)
print(f"{ds.name}: {ds.df.shape[0]:,} rows x {ds.df.shape[1]} columns | split = {ds.split}")
print(ds.notes)
display(ds.df["label"].value_counts().to_frame("rows"))
ds.df.head()

## 2. Split and prepare the data

First the 70/30 split. How it is done depends on the dataset: **stratified random** (IoT-23, CIC-IDS2017), **chronological** (SDN: a random split would put near-copies of the same flow on both sides), or **grouped by incident** (GUIDE).

Then the paper's preprocessing (Section IV-B). Everything is learned on TRAIN and applied to TEST.

| Paper step | What the code does |
|---|---|
| (1) remove duplicated / >0.95-correlated features | **Yes**: constant columns, duplicate columns, then one of each correlated numeric pair |
| (2) one-hot / label encoding | **Not here.** CTGAN, TVAE and CTAB-GAN+ model categorical columns natively, and the XGBoost classifier ordinal-encodes them |
| (3) IQR outlier removal, then z-score | **Optional, off by default.** On IoT-23 the IQR filter deletes **34% of malicious flows but only 7% of benign flows**: attacks *are* the outliers. Removing them would remove what the generator must learn. Turn it on to test it; the removed fractions per class are printed |
| (4) median / mode imputation | **Yes** |

If you keep the defaults, the paper's text (which says all four steps are applied) should say what was actually done.

In [ ]:
train_raw, test_raw = A.prep.split(ds)
P = A.prep.prepare(ds, train_raw, test_raw, CORR_THRESHOLD, REMOVE_OUTLIERS, ZSCORE)
train, test = P.train, P.test
FEATURES, NUMERICAL, CATEGORICAL = P.features, P.numerical, P.categorical
n_select = min(N_SELECT or ds.n_select, len(FEATURES) - 1)

print("train", train.shape, "| test", test.shape, "| candidate features:", len(FEATURES),
      f"({len(NUMERICAL)} numeric, {len([c for c in FEATURES if c in CATEGORICAL])} categorical)")
print("dropped constant        :", P.info["constant"])
print("dropped duplicate cols  :", P.info["duplicate_columns"])
print(f"dropped correlated >{CORR_THRESHOLD}:", P.info["correlated"])
if P.info["outliers_removed"]:
    print("fraction of TRAIN rows removed by IQR filter, per class:", P.info["outliers_removed"])
pd.DataFrame({"train": train["label"].value_counts(), "test": test["label"].value_counts()})

## 3. Rank features with five techniques (TRAIN only)

| Family | How it works | Technique |
|---|---|---|
| **Filter** | Scores each feature with a statistic, without training a model. Fast, ignores interactions | **ANOVA F-test** (numeric only), **Chi-square → Cramér's V** (numeric features are binned first), **Mutual information** (also catches non-linear links) |
| **Wrapper** | Repeatedly trains a model on feature subsets (here: drop the weakest feature, retrain, repeat) | **RFE** with a random forest |
| **Embedded** | Selection happens inside model training | **Random-forest importance** |

Higher score = better for every column below (RFE is shown as the negated elimination rank).

In [ ]:
scores = A.featsel.rank_features(P, n_select)
scores.sort_values("mutual_info (filter)", ascending=False).round(4)

## 4. Voting

Each technique gives **one vote** to each of its top `n_select` features, so a feature can get at most 5 votes. Features are sorted by votes; ties are broken by **average rank** (lower = better). This is a rule we are *testing*, not a guarantee the subset is optimal.

In [ ]:
vote_table, SELECTED = A.featsel.vote(scores, n_select)
print("Selected features:", SELECTED)
vote_table[["votes", "avg_rank", "selected"] + [c for c in vote_table.columns if c.startswith("vote_")]]

In [ ]:
vote_table["votes"].iloc[::-1].plot.barh(color=["#2a7" if s else "#bbb" for s in vote_table["selected"].iloc[::-1]],
                                          figsize=(6, max(3, 0.3 * len(vote_table))), title="Votes per feature (green = selected)")
plt.xlabel("votes (max 5)"); plt.tight_layout(); plt.show()

## 5. Train the generators

For every generator we train **twice** (same settings and seed, so the only difference is the column set):
* **all features** – every candidate feature + `label`
* **voted top-N** – only the selected features + `label`

Each synthetic table has as many rows as the training table (same size and class mix; the paper's *class-balanced* sampling is not implemented yet). `train_seconds` covers fitting only. CTAB-GAN+ and TabDDPM-style models are slow on many columns, so start with a small `EPOCHS` to check things run.

In [ ]:
ALL_COLS, SEL_COLS = FEATURES + ["label"], SELECTED + ["label"]
FEATURE_SETS = {"all features": ALL_COLS, "voted top-N": SEL_COLS}

synthetic, fit_stats = {}, {}
for seed in SEEDS:
    for g in GENERATORS:
        for fs, cols in FEATURE_SETS.items():
            print(f"[{time.strftime('%H:%M:%S')}] {A.generators.DISPLAY[g]} | {fs} ({len(cols)-1} features) | seed {seed} ...", flush=True)
            syn, st = A.generators.fit_and_sample(g, train, cols, CATEGORICAL, seed=seed, epochs=EPOCHS)
            synthetic[(g, fs, seed)], fit_stats[(g, fs, seed)] = syn, st
            print(f"    done in {st['train_seconds']:.0f}s, peak GPU {st['peak_gpu_mem_gb']:.2f} GB", flush=True)

## 6. Compare synthetic vs real (against the held-out TEST table)

We never compare row 1 with row 1. We compare **distributions**, **relationships** and **usefulness**. Every table is scored on the **same columns** (the voted ones + `label`) so the two feature sets are compared fairly.

* **A. Column distributions** (0–1, higher = more similar): `KSComplement` for numeric, `TVComplement` for categorical columns → `sim_numerical`, `sim_categorical`
* **B. Relationships** (0–1): `CorrelationSimilarity` for numeric pairs, `ContingencySimilarity` for categorical pairs → `sim_relationships` (pairs involving a constant synthetic column are skipped)
* **C. Utility (TSTR)**: train **XGBoost on the synthetic table only**, test on real test data → `TSTR_F1` (macro) and `TSTR_AUC` (macro one-vs-rest). **TRTR** does the same with the real training data, as the reference. A class the synthetic table never contains cannot be predicted (F1 = 0 for it).

A similarity of 0.90 is not "90% accuracy", and none of this measures privacy.

In [ ]:
COMMON = SEL_COLS
CLASSES = sorted(set(train["label"]) | set(test["label"]))

trtr = A.evaluate.utility(train, test, COMMON, CATEGORICAL, CLASSES)
ref_sim, _ = A.evaluate.similarity(test, train, COMMON, CATEGORICAL)
print("TRTR (train on REAL train, test on REAL test):", {k: round(v, 3) for k, v in trtr.items()})
print("real train vs real test similarity           :", {k: round(float(v), 3) for k, v in ref_sim.items()})

rows, per_column = [], {}
for (g, fs, seed), syn in synthetic.items():
    sim, pc = A.evaluate.similarity(test, syn, COMMON, CATEGORICAL)
    per_column[(g, fs, seed)] = pc
    u = A.evaluate.utility(syn, test, COMMON, CATEGORICAL, CLASSES)
    rows.append({"generator": g, "feature_set": fs, "seed": seed, "n_features": len(FEATURE_SETS[fs]) - 1,
                 **sim, "TSTR_F1": u["f1_macro"], "TSTR_AUC": u["roc_auc"],
                 "train_seconds": fit_stats[(g, fs, seed)]["train_seconds"],
                 "peak_gpu_gb": fit_stats[(g, fs, seed)]["peak_gpu_mem_gb"]})
results = pd.DataFrame(rows)
results.round(3)

### Summary (mean over seeds; ± std appears when you use more than one seed)

In [ ]:
summary = (results.drop(columns="seed").groupby(["generator", "feature_set"]).agg(["mean", "std"]).round(3))
summary

## 7. Look at it, not just the numbers

Correlation heatmaps of the numeric selected columns: real test on the left, then each generator trained on the voted features. Similar colours = relationships preserved.

In [ ]:
num_common = [c for c in COMMON if c in NUMERICAL]
seed0 = SEEDS[0]
if len(num_common) >= 2:
    panels = [("REAL (test)", test)] + [(A.generators.DISPLAY[g], synthetic[(g, "voted top-N", seed0)]) for g in GENERATORS]
    fig, axes = plt.subplots(1, len(panels), figsize=(5 * len(panels), 4.5), squeeze=False)
    for ax, (title, frame) in zip(axes[0], panels):
        im = ax.imshow(frame[num_common].corr(), vmin=-1, vmax=1, cmap="coolwarm")
        ax.set_xticks(range(len(num_common))); ax.set_xticklabels(num_common, rotation=90)
        ax.set_yticks(range(len(num_common))); ax.set_yticklabels(num_common); ax.set_title(title)
    fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.8); plt.show()
else:
    print("Fewer than two numeric selected columns: no heatmap.")

In [ ]:
# per-column similarity: which individual columns did each model reproduce well?
pd.DataFrame({f"{A.generators.DISPLAY[g]} | {fs}": per_column[(g, fs, seed0)]
              for g in GENERATORS for fs in FEATURE_SETS}).round(3)

## 8. Save outputs

Everything goes to `results_pipeline/<dataset>/`.

In [ ]:
out = PROJECT / "results_pipeline" / ds.key
out.mkdir(parents=True, exist_ok=True)
scores.to_csv(out / "feature_scores.csv"); vote_table.to_csv(out / "voting_table.csv")
pd.Series(SELECTED, name="feature").to_csv(out / "selected_features.csv", index=False)
results.to_csv(out / "comparison.csv", index=False); summary.to_csv(out / "summary.csv")
for (g, fs, seed), syn in synthetic.items():
    syn.to_csv(out / f"synthetic_{g}_{fs.split()[0]}_seed{seed}.csv", index=False)
config = {"dataset": ds.key, "generators": GENERATORS, "epochs": EPOCHS, "seeds": SEEDS, "n_select": n_select,
          "corr_threshold": CORR_THRESHOLD, "remove_outliers": REMOVE_OUTLIERS, "zscore": ZSCORE, "split": ds.split,
          "selected_features": SELECTED, "prep_info": {k: v for k, v in P.info.items()},
          "trtr": trtr, "real_vs_real_similarity": {k: float(v) for k, v in ref_sim.items()}, "notes": ds.notes}
(out / "run_config.json").write_text(json.dumps(config, indent=2, default=str))
print("Saved to", out.relative_to(PROJECT))

## 9. Log to the paper results workbook

Appends one row per generator × feature set × seed to `paper_results/experiment_log.csv` and rebuilds `paper_results/AdaptSynth_results.xlsx`. Re-running with identical settings replaces the earlier numbers; changing a data-preparation switch adds a `config_tag` so those runs stay separate.

In [ ]:
if LOG_TO_PAPER_RESULTS:
    from results_log import log_result
    from build_results_workbook import build
    tag = "+".join(t for t in ["iqr" if REMOVE_OUTLIERS else "", "zscore" if ZSCORE else "",
                               f"corr{CORR_THRESHOLD}" if CORR_THRESHOLD != 0.95 else "",
                               f"cap{ds.info['max_rows_per_class']}" if ds.info.get("max_rows_per_class") not in (None, 2000) else ""] if t)
    for r in results.itertuples():
        log_result(dataset=ds.name, generator=A.generators.DISPLAY[r.generator], feature_set=r.feature_set, seed=r.seed,
                   n_features=r.n_features, epochs=None if r.generator == "smote" else EPOCHS, classifier="XGBoost",
                   config_tag=tag, tstr_f1_macro=r.TSTR_F1, tstr_roc_auc=r.TSTR_AUC,
                   trtr_f1_macro=trtr["f1_macro"], trtr_roc_auc=trtr["roc_auc"],
                   sim_numerical=r.sim_numerical, sim_categorical=r.sim_categorical, sim_relationships=r.sim_relationships,
                   train_seconds=r.train_seconds, peak_gpu_mem_gb=r.peak_gpu_gb,
                   source="adaptsynth_pipeline.ipynb", notes=ds.notes)
    build()

## Adding another model (TabDDPM, GVAE, ...)

A generator is a class with two methods. Register it in `src/adaptsynth/generators.py` (copy the SMOTE class as a template) and it appears in the `GENERATORS` list:

```python
from adaptsynth.generators import register

@register("tabddpm", "TabDDPM")            # key used in GENERATORS, name shown in tables and the workbook
class TabDDPM:
    def fit(self, data, categorical, seed, epochs):
        # data: DataFrame, last column is "label"; categorical: list of categorical column names
        ...
        return self
    def sample(self, n):
        ...                                  # return a DataFrame with the same columns as `data`
```

Then set `GENERATORS = ["ctabgan_plus", "tabddpm"]`. Training time, GPU memory, similarity, TSTR/TRTR and the workbook rows come for free.

## How to read the results (and what NOT to conclude)

* If *voted top-N* scores about the same as *all features* on the same columns while training faster, the feature vote kept what matters.
* A single seed can differ by a few hundredths from noise. Use `SEEDS = [42, 43, 44]` before claiming one model is better.
* TSTR close to TRTR means the synthetic data is nearly as useful as real data for this classifier. If TRTR itself is near chance, the dataset has no learnable signal and nothing downstream is meaningful.
* Similarity scores measure statistical fidelity only: not privacy, and (except TSTR) not usefulness.